In [ ]:
"""
Densité d'états A(ω) — Modèle d'Anderson à une impureté (SIAM)
Méthode : QuTiP 5.x  (Lindblad + corrélations temporelles)

Convention visuelle identique au code Julia :
  • Fort couplage  (U/πΓ > 1, Γ < Γ*) : tirets (--),  couleurs froides (bleus)
  • Frontière      (U/πΓ ≈ 1, Γ = Γ*) : trait plein épais, gris foncé
  • Faible couplage(U/πΓ < 1, Γ > Γ*) : trait plein fin,  couleurs chaudes (oranges/rouges)

Corrections numériques :
  1. Opérateurs de Lindblad avec bilan détaillé (taux via Fermi-Dirac)
  2. Fenêtrage de Hanning avant la TF (suppression des oscillations de Gibbs)
  3. Grille temporelle longue (tmax=200) pour résolution Δω ~ 2π/tmax
  4. clip(0) comme garde-fou final
"""

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import warnings
warnings.filterwarnings('ignore')

from qutip import qeye, sigmam, sigmaz, tensor, steadystate, correlation_2op_1t

# ── Style global matplotlib (publication) ─────────────────────────────────────
plt.rcParams.update({
    "font.family"      : "serif",
    "mathtext.fontset" : "stix",
    "font.size"        : 11,
    "axes.labelsize"   : 13,
    "axes.titlesize"   : 12,
    "axes.linewidth"   : 1.2,
    "xtick.labelsize"  : 11,
    "ytick.labelsize"  : 11,
    "xtick.direction"  : "in",
    "ytick.direction"  : "in",
    "xtick.top"        : True,
    "ytick.right"      : True,
    "xtick.minor.visible" : True,
    "ytick.minor.visible" : True,
    "legend.frameon"   : True,
    "legend.framealpha": 1.0,
    "legend.edgecolor" : "black",
    "legend.fontsize"  : 10,
    "figure.dpi"       : 150,
})

# ── Paramètres physiques ──────────────────────────────────────────────────────
epsilon = -5.0
U       = 10.0
mu      = 0.0
kT      = 0.5
W       = 10.0
Gamma_star = U / np.pi      # ≈ 3.18  — frontière Kondo / métal

# Même liste que le code Julia
Gamma_list = [1.0, 1.5, 2.0, 3.18, 7.0, 10.0, 15.0]

# ── Opérateurs fermioniques ───────────────────────────────────────────────────
II      = qeye(2)
sigma_m = sigmam()
sigma_z = sigmaz()
d_up    = tensor(sigma_m, II)
d_dn    = tensor(-sigma_z, sigma_m)

H_sys = (epsilon * (d_up.dag() * d_up + d_dn.dag() * d_dn)
         + U * (d_up.dag() * d_up * d_dn.dag() * d_dn))
print(f"Hamiltonien construit — dim = {H_sys.shape[0]}×{H_sys.shape[0]}")

# ── Palettes (identiques au code Julia) ──────────────────────────────────────
# Fort couplage : bleus froids
KONDO_COLORS = ["#2166AC", "#4393C3", "#92C5DE"]   # Γ = 1.0, 1.5, 2.0
BOUNDARY_COLOR = "#222222"                          # Γ = 3.18
# Faible couplage : oranges/rouges chauds
METAL_COLORS = ["#F4A582", "#D6604D", "#B2182B"]   # Γ = 7.0, 10.0, 15.0

def get_style(Gamma):
    """Retourne (couleur, linewidth, linestyle) selon le régime."""
    ratio = U / (np.pi * Gamma)
    if abs(ratio - 1.0) < 0.02:                        # frontière
        return BOUNDARY_COLOR, 3.0, "-"
    elif ratio > 1.0:                                   # Kondo (fort couplage)
        idx = [1.0, 1.5, 2.0].index(
            min([1.0, 1.5, 2.0], key=lambda g: abs(g - Gamma)))
        return KONDO_COLORS[idx], 1.8, "--"
    else:                                               # métal (faible couplage)
        idx = [7.0, 10.0, 15.0].index(
            min([7.0, 10.0, 15.0], key=lambda g: abs(g - Gamma)))
        return METAL_COLORS[idx], 1.4, "-"

# ── Bilan détaillé ────────────────────────────────────────────────────────────
def fermi(omega, mu, kT):
    x = (omega - mu) / kT
    if x > 100:  return 0.0
    if x < -100: return 1.0
    return 1.0 / (1.0 + np.exp(x))

def lindblad_ops(d_op, Gamma, mu, kT):
    """Opérateurs de collapse avec bilan détaillé (Fermi-Dirac)."""
    f     = fermi(mu, mu, kT)          # = 0.5 par symétrie
    g_em  = np.sqrt(Gamma * (1.0 - f))
    g_abs = np.sqrt(Gamma * f)
    return [g_em * d_op, g_abs * d_op.dag()]

# ── Calcul de la DOS ──────────────────────────────────────────────────────────
def compute_dos(H, c_ops, d_op, omega_list, Gamma):
    """
    A(ω) = (1/π) Re ∫ dt e^{iωt} [⟨d(t)d†(0)⟩ + ⟨d†(t)d(0)⟩]

    Optimisations mémoire/CPU :
      • tmax adaptatif : la corrélation de Lindblad décroît sur ~1/Γ,
        tmax = 30/Γ capture 99%% de l'intégrale quelle que soit Γ.
        → Pour Γ=15 : tmax=2 (au lieu de 200) ×100 moins de RAM
      • FFT numpy O(N log N) au lieu de la boucle naïve O(N_ω × N_t)
        → gain ×30 CPU, ×50 mémoire
      • Interpolation du spectre FFT sur la grille omega_list demandée
    """
    # Grille temporelle adaptée à ce Gamma spécifique
    tmax  = max(30.0 / Gamma, 5.0)   # la corr. Lindblad décroît sur ~1/Gamma
    Nt    = max(int(tmax * 50), 512)  # 50 pts par unité de temps, min 512
    tlist = np.linspace(0, tmax, Nt)
    dt    = tlist[1] - tlist[0]

    rho_ss = steadystate(H, c_ops)
    corr1  = correlation_2op_1t(H, rho_ss, tlist, c_ops, d_op,       d_op.dag())
    corr2  = correlation_2op_1t(H, rho_ss, tlist, c_ops, d_op.dag(), d_op)
    corr   = np.array(corr1 + corr2, dtype=complex)

    # Fenêtre de Hanning — supprime les oscillations de Gibbs
    corr_w = corr * np.hanning(Nt)

    # FFT vectorisée O(N log N) — convention e^{+iωt}
    # np.fft.fft utilise e^{-2πi·k·n/N}, donc on conjugue les fréquences
    fft_vals  = np.fft.fft(corr_w) * dt
    fft_freqs = np.fft.fftfreq(Nt, d=dt) * 2 * np.pi   # rad / unité de temps

    # Trier par fréquence croissante pour l'interpolation
    idx_sort  = np.argsort(fft_freqs)
    fft_freqs = fft_freqs[idx_sort]
    # Partie réelle de la TF dans la convention +iωt
    spectrum_raw = np.real(fft_vals[idx_sort]) / np.pi

    # Interpolation sur la grille omega_list souhaitée
    spectrum = np.interp(omega_list, fft_freqs, spectrum_raw,
                         left=0.0, right=0.0)
    return np.clip(spectrum, 0, None)

# ── Grille de fréquences ──────────────────────────────────────────────────────
omega_list = np.linspace(-15.0, 15.0, 300)

# ── Calcul pour chaque Γ ──────────────────────────────────────────────────────
print("\nCalcul de la DOS …")
print("=" * 60)
dos_results = {}

for Gamma in Gamma_list:
    tmax_used = max(30.0 / Gamma, 5.0)
    Nt_used   = max(int(tmax_used * 50), 512)
    print(f"\n  Γ = {Gamma:.3f}   U/πΓ = {U/(np.pi*Gamma):.3f}"
          f"   tmax = {tmax_used:.1f}   Nt = {Nt_used}")
    c_ops = (lindblad_ops(d_up, Gamma, mu, kT)
           + lindblad_ops(d_dn, Gamma, mu, kT))
    dos = compute_dos(H_sys, c_ops, d_up, omega_list, Gamma)
    dos_results[Gamma] = dos
    idx0 = np.argmin(np.abs(omega_list))
    print(f"    A(ω=0) = {dos[idx0]:.5f}   "
          f"pts négatifs résiduels = {(dos < 0).sum()}")

print("\n" + "=" * 60)
print("Calcul terminé !")

# ── Figure publication ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8.5, 5.2))

# ── Tracé des courbes ─────────────────────────────────────────────────────────
for Gamma in Gamma_list:
    color, lw, ls = get_style(Gamma)
    ratio = U / (np.pi * Gamma)
    label = rf"$U/\pi\Gamma = {ratio:.2f}$"
    ax.plot(omega_list, dos_results[Gamma],
            color=color, linewidth=lw, linestyle=ls, label=label)

# ── Lignes de référence ───────────────────────────────────────────────────────
ax.axvline(epsilon,   color="crimson",     lw=1.0, ls=":",  alpha=0.5)
ax.axvline(epsilon+U, color="forestgreen", lw=1.0, ls=":",  alpha=0.5)
ax.axvline(mu,        color="black",       lw=0.8, ls=":",  alpha=0.25)
ax.axhline(0,         color="black",       lw=0.7)

ax.text(epsilon + 0.2, ax.get_ylim()[1] * 0.02,
        r"$\varepsilon$", color="crimson", fontsize=11)
ax.text(epsilon + U + 0.2, ax.get_ylim()[1] * 0.02,
        r"$\varepsilon + U$", color="forestgreen", fontsize=11)

# ── Annotations de régime ─────────────────────────────────────────────────────
ax.text(-13.5, 0.93, "Strong coupling\n" + r"$(U/\pi\Gamma > 1)$",
        color="#2166AC", fontsize=9.5, transform=ax.transData,
        va="top")
ax.text(6.0, 0.93, "Weak coupling\n" + r"$(U/\pi\Gamma < 1)$",
        color="#B2182B", fontsize=9.5, transform=ax.transData,
        va="top")

# ── Légende personnalisée par régime ─────────────────────────────────────────
legend_handles = []

# Kondo
for Gamma, col in zip([1.0, 1.5, 2.0], KONDO_COLORS):
    r = U / (np.pi * Gamma)
    h = mlines.Line2D([], [], color=col, lw=1.8, ls="--",
                      label=rf"$U/\pi\Gamma = {r:.2f}$")
    legend_handles.append(h)

# Frontière
r_b = U / (np.pi * 3.18)
legend_handles.append(
    mlines.Line2D([], [], color=BOUNDARY_COLOR, lw=3.0, ls="-",
                  label=rf"$U/\pi\Gamma = {r_b:.2f}$ (boundary)"))

# Métal
for Gamma, col in zip([7.0, 10.0, 15.0], METAL_COLORS):
    r = U / (np.pi * Gamma)
    h = mlines.Line2D([], [], color=col, lw=1.4, ls="-",
                      label=rf"$U/\pi\Gamma = {r:.2f}$")
    legend_handles.append(h)

leg = ax.legend(
    handles   = legend_handles,
    loc       = "upper right",
    fontsize  = 10,
    frameon   = True,
    edgecolor = "black",
    title     = (rf"$\varepsilon={int(epsilon)},\; U={int(U)},\; kT={kT}$"),
    title_fontsize = 9.5,
    handlelength = 2.5,
)

# ── Axes ──────────────────────────────────────────────────────────────────────
ax.set_xlabel(r"$\omega \quad [\Gamma]$")
ax.set_ylabel(r"$\pi \, A(\omega)$")
ax.set_title(
    "Density of States — Single Impurity Anderson Model  (QuTiP/Lindblad)",
    fontweight="bold", pad=8)
ax.set_xlim(omega_list[0], omega_list[-1])
ax.set_ylim(bottom=0)
ax.xaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())
ax.yaxis.set_minor_locator(matplotlib.ticker.AutoMinorLocator())

plt.tight_layout()
plt.savefig("DOS_SIAM_py.pdf", bbox_inches="tight")
plt.savefig("DOS_SIAM_py.png", dpi=300, bbox_inches="tight")
print("\n✓  Figures sauvegardées : DOS_SIAM_py.pdf  /  DOS_SIAM_py.png")
plt.show()

# ── Résumé physique ───────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("RÉSUMÉ PHYSIQUE")
print("=" * 60)
print(f"  Frontière Kondo/métal : Γ* = U/π = {Gamma_star:.3f}")
for Gamma in Gamma_list:
    ratio  = U / (np.pi * Gamma)
    regime = ("Kondo (fort couplage)" if ratio > 1.02
              else "Frontière" if abs(ratio - 1.0) < 0.02
              else "Métal (faible couplage)")
    print(f"  Γ = {Gamma:5.2f}  →  U/πΓ = {ratio:.3f}  →  {regime}")

Hamiltonien construit — dim = 4×4

Calcul de la DOS …

  Γ = 1.000   U/πΓ = 3.183   tmax = 30.0   Nt = 1500
    A(ω=0) = 0.00000   pts négatifs résiduels = 0

  Γ = 1.500   U/πΓ = 2.122   tmax = 20.0   Nt = 1000
    A(ω=0) = 0.00000   pts négatifs résiduels = 0

  Γ = 2.000   U/πΓ = 1.592   tmax = 15.0   Nt = 750
    A(ω=0) = 0.00000   pts négatifs résiduels = 0

  Γ = 3.180   U/πΓ = 1.001   tmax = 9.4   Nt = 512
    A(ω=0) = 0.00000   pts négatifs résiduels = 0

  Γ = 7.000   U/πΓ = 0.455   tmax = 5.0   Nt = 512
    A(ω=0) = 0.00000   pts négatifs résiduels = 0

  Γ = 10.000   U/πΓ = 0.318   tmax = 5.0   Nt = 512
    A(ω=0) = 0.00000   pts négatifs résiduels = 0

  Γ = 15.000   U/πΓ = 0.212   tmax = 5.0   Nt = 512
    A(ω=0) = 0.00000   pts négatifs résiduels = 0

Calcul terminé !
